In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Представлення квантових комп'ютерів для транспілятора


<details>
<summary><b>Package versions</b></summary>

The code on this page was developed using the following requirements.
We recommend using these versions or newer.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
Щоб перетворити абстрактну схему на ISA-схему, яка може виконуватись на конкретному QPU (квантовому процесорі), транспілятору потрібна певна інформація про QPU. Ця інформація знаходиться в двох місцях: об'єкті `BackendV2` (або застарілому `BackendV1`), до якого ти плануєш подавати завдання, та атрибуті `Target` бекенду.

- [`Target`](../api/qiskit/qiskit.transpiler.Target) містить усі відповідні обмеження пристрою, такі як його рідні базові вентилі, зв'язність кубітів та інформацію про імпульси або часування.
- [`Backend`](../api/qiskit/qiskit.providers.BackendV2) за замовчуванням має `Target`, містить додаткову інформацію — наприклад, [`InstructionScheduleMap`](https://docs.quantum.ibm.com/api/qiskit/1.4/qiskit.pulse.InstructionScheduleMap), — та надає інтерфейс для подачі завдань з квантовими схемами.

Ти також можеш явно надати інформацію для використання транспілятором, наприклад, якщо маєш конкретний сценарій використання, або якщо вважаєш, що ця інформація допоможе транспілятору згенерувати більш оптимізовану схему.

Точність, з якою транспілятор створює найбільш відповідну схему для конкретного апаратного забезпечення, залежить від того, скільки інформації про свої обмеження має `Target` або `Backend`.

> **Note:** Оскільки багато базових алгоритмів транспіляції є стохастичними, немає гарантії, що буде знайдено кращу схему.

На цій сторінці наведено кілька прикладів передачі інформації про QPU транспілятору. Ці приклади використовують ціль із імітаційного бекенду [`FakeSherbrooke`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/fake-provider-fake-sherbrooke#fakesherbrooke).

<span id="default-config"></span>
## Конфігурація за замовчуванням
Найпростіший спосіб використання транспілятора — надати всю інформацію про QPU, передавши `Backend` або `Target`. Щоб краще зрозуміти, як працює транспілятор, побудуй схему та транспілюй її з різною інформацією, як показано нижче.

Імпортуй необхідні бібліотеки та ініціалізуй QPU:
Щоб перетворити абстрактну схему на ISA-схему, яка може виконуватись на конкретному процесорі, транспілятору потрібна певна інформація про процесор. Зазвичай ця інформація зберігається у [`Backend`](https://docs.quantum.ibm.com/api/qiskit/qiskit.providers.Backend#backend) або [`Target`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.Target#target), що передаються транспілятору, і ніяка додаткова інформація не потрібна. Однак ти також можеш явно надати інформацію для використання транспілятором, наприклад, якщо маєш конкретний сценарій використання, або якщо вважаєш, що ця інформація допоможе транспілятору згенерувати більш оптимізовану схему.

У цьому розділі наведено кілька прикладів передачі інформації транспілятору. Ці приклади використовують ціль із імітаційного бекенду [`FakeSherbrooke`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/fake-provider-fake-sherbrooke#fakesherbrooke).

In [1]:
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke

backend = FakeSherbrooke()
target = backend.target

Схема прикладу використовує екземпляр [`efficient_su2`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.efficient_su2) з бібліотеки схем Qiskit.

In [2]:
from qiskit.circuit.library import efficient_su2

qc = efficient_su2(12, entanglement="circular", reps=1)

qc.draw("mpl")

<Image src="../docs/images/guides/represent-quantum-computers/extracted-outputs/97f9acc1-ac53-4025-b413-485777932a9b-0.svg" alt="Output of the previous code cell" />

![Вихідні дані попереднього блоку коду](../docs/images/guides/represent-quantum-computers/extracted-outputs/97f9acc1-ac53-4025-b413-485777932a9b-0.svg)

У цьому прикладі використовуються параметри за замовчуванням для транспіляції до `target` бекенду, що надає всю необхідну інформацію для перетворення схеми на таку, що буде виконуватися на бекенді.

In [3]:
from qiskit.transpiler import generate_preset_pass_manager

pass_manager = generate_preset_pass_manager(
    optimization_level=1, target=target, seed_transpiler=12345
)
qc_t_target = pass_manager.run(qc)
qc_t_target.draw("mpl", idle_wires=False, fold=-1)

<Image src="../docs/images/guides/represent-quantum-computers/extracted-outputs/4b81fb9d-d199-45c5-b119-c1f0b973afe9-0.svg" alt="Output of the previous code cell" />

![Вихідні дані попереднього блоку коду](../docs/images/guides/represent-quantum-computers/extracted-outputs/4b81fb9d-d199-45c5-b119-c1f0b973afe9-0.svg)

Цей приклад використовується в наступних розділах цього матеріалу для ілюстрації того, що карта зв'язності та базові вентилі є ключовими елементами інформації, що передається транспілятору для оптимальної побудови схеми. QPU зазвичай може вибирати параметри за замовчуванням для іншої інформації, що не передається, наприклад для часування та планування.

## Карта зв'язності

Карта зв'язності — це граф, що показує, які кубіти з'єднані та мають між собою двокубітні вентилі. Іноді цей граф є напрямленим, тобто двокубітні вентилі можуть діяти лише в одному напрямку. Однак транспілятор завжди може змінити напрям вентиля, додаючи додаткові однокубітні вентилі. Абстрактна квантова схема завжди може бути представлена на цьому графі, навіть якщо його зв'язність обмежена, шляхом введення вентилів SWAP для переміщення квантової інформації.

Кубіти з наших абстрактних схем називаються _віртуальними кубітами_, а ті на карті зв'язності — _фізичними кубітами_. Транспілятор забезпечує відображення між віртуальними та фізичними кубітами. Один із перших кроків транспіляції — етап _розміщення_ — виконує це відображення.

> **Note:** Хоча етап маршрутизації взаємопов'язаний з етапом _розміщення_ — який вибирає фактичні кубіти — за замовчуванням цей матеріал розглядає їх як окремі етапи для простоти. Поєднання маршрутизації та розміщення називається _відображенням кубітів_. Докладніше про ці етапи дивись у матеріалі [Етапи транспілятора](transpiler-stages).

Передай ключовий аргумент `coupling_map`, щоб побачити його вплив на транспілятор:

In [4]:
coupling_map = target.build_coupling_map()

pass_manager = generate_preset_pass_manager(
    optimization_level=0, coupling_map=coupling_map, seed_transpiler=12345
)
qc_t_cm_lv0 = pass_manager.run(qc)
qc_t_cm_lv0.draw("mpl", idle_wires=False, fold=-1)

<Image src="../docs/images/guides/represent-quantum-computers/extracted-outputs/ec354bee-e06b-42ea-a117-6c1a9308ca73-0.svg" alt="Output of the previous code cell" />

![Вихідні дані попереднього блоку коду](../docs/images/guides/represent-quantum-computers/extracted-outputs/ec354bee-e06b-42ea-a117-6c1a9308ca73-0.svg)

Як показано вище, було вставлено кілька вентилів SWAP (кожен з яких складається з трьох вентилів CX), що призведе до великої кількості помилок на поточних пристроях. Щоб побачити, які кубіти вибрані на фактичній топології кубітів, скористайся `plot_circuit_layout` з Qiskit Visualizations:

In [5]:
from qiskit.visualization import plot_circuit_layout

plot_circuit_layout(qc_t_cm_lv0, backend, view="physical")

<Image src="../docs/images/guides/represent-quantum-computers/extracted-outputs/9be74535-ed36-4d51-afeb-ee53c3f8a046-0.svg" alt="Output of the previous code cell" />

![Вихідні дані попереднього блоку коду](../docs/images/guides/represent-quantum-computers/extracted-outputs/9be74535-ed36-4d51-afeb-ee53c3f8a046-0.svg)

Це показує, що наші віртуальні кубіти 0–11 були тривіально відображені на лінію фізичних кубітів 0–11. Повернімось до параметрів за замовчуванням (`optimization_level=1`), що використовує `VF2Layout` якщо потрібна будь-яка маршрутизація.

In [6]:
pass_manager = generate_preset_pass_manager(
    optimization_level=1, coupling_map=coupling_map, seed_transpiler=12345
)
qc_t_cm_lv1 = pass_manager.run(qc)
qc_t_cm_lv1.draw("mpl", idle_wires=False, fold=-1)

<Image src="../docs/images/guides/represent-quantum-computers/extracted-outputs/8035fd05-f7cd-4151-b19a-4968202246e6-0.svg" alt="Output of the previous code cell" />

![Вихідні дані попереднього блоку коду](../docs/images/guides/represent-quantum-computers/extracted-outputs/8035fd05-f7cd-4151-b19a-4968202246e6-0.svg)

Тепер вентилі SWAP не вставляються, а вибрані фізичні кубіти збігаються з тими, що при використанні класу `target`.

In [7]:
from qiskit.visualization import plot_circuit_layout

plot_circuit_layout(qc_t_cm_lv1, backend, view="physical")

<Image src="../docs/images/guides/represent-quantum-computers/extracted-outputs/25d9fac3-abda-4b2d-81b4-351dc0772722-0.svg" alt="Output of the previous code cell" />

![Вихідні дані попереднього блоку коду](../docs/images/guides/represent-quantum-computers/extracted-outputs/25d9fac3-abda-4b2d-81b4-351dc0772722-0.svg)

Тепер розміщення утворює кільце. Оскільки це розміщення відповідає зв'язності схеми, вентилі SWAP відсутні, що забезпечує значно кращу схему для виконання.

## Базові вентилі

Кожен квантовий комп'ютер підтримує обмежений набір інструкцій, що називається його _базовими вентилями_. Кожен вентиль у схемі має бути перетворений на елементи цього набору. Цей набір має складатися з однокубітних і двокубітних вентилів, що утворюють універсальний набір вентилів, тобто будь-яка квантова операція може бути розкладена на ці вентилі. Це виконується за допомогою [BasisTranslator](../api/qiskit/qiskit.transpiler.passes.BasisTranslator), а базові вентилі можна вказати як ключовий аргумент транспілятора для надання цієї інформації.

In [8]:
basis_gates = list(target.operation_names)
print(basis_gates)

['sx', 'switch_case', 'x', 'if_else', 'measure', 'for_loop', 'delay', 'ecr', 'id', 'reset', 'rz']


The default single-qubit gates on _ibm_sherbrooke_ are `rz`, `x`, and `sx`, and the default two-qubit gate is `ecr` (echoed cross-resonance). CX gates are constructed from `ecr` gates, so on some QPUs `ecr` is specified as the two-qubit basis gate, while on others `cx` is the default. The `ecr` gate is the _entangling_ part of the `cx` gate. In addition to the control gates, there are also `delay` and `measurement` instructions.


<Admonition type="note">
    QPUs have default basis gates, but you can choose whatever gates you want, as long as you provide the instruction or add pulse gates (see [Create transpiler passes.](custom-transpiler-pass)) The default basis gates are those that calibrations have been done for on the QPU, so no further instruction/pulse gates need to be provided. For example, on some QPUs `cx` is the default two-qubit gate and `ecr` on others. See the list of possible [native gates and operations](/docs/guides/qpu-information#native-gates) for more details.
</Admonition>

In [9]:
pass_manager = generate_preset_pass_manager(
    optimization_level=1,
    coupling_map=coupling_map,
    basis_gates=basis_gates,
    seed_transpiler=12345,
)
qc_t_cm_bg = pass_manager.run(qc)
qc_t_cm_bg.draw("mpl", idle_wires=False, fold=-1)

<Image src="../docs/images/guides/represent-quantum-computers/extracted-outputs/313e4743-0.svg" alt="Output of the previous code cell" />

Однокубітні вентилі за замовчуванням на _ibm_sherbrooke_ — це `rz`, `x` та `sx`, а двокубітний вентиль за замовчуванням — `ecr` (відлунний перехресний резонанс). Вентилі CX будуються з вентилів `ecr`, тому на деяких QPU `ecr` вказується як двокубітний базовий вентиль, тоді як на інших за замовчуванням використовується `cx`. Вентиль `ecr` є _заплутуючою_ частиною вентиля `cx`. Крім керуючих вентилів, є також інструкції `delay` та `measurement`.

> **Note:** QPU мають базові вентилі за замовчуванням, але ти можеш вибрати будь-які вентилі, за умови надання інструкції або додавання імпульсних вентилів (дивись [Створення проходів транспілятора.](custom-transpiler-pass)). Базові вентилі за замовчуванням — це ті, для яких виконані калібрування на QPU, тому не потрібно надавати додаткових інструкцій/імпульсних вентилів. Наприклад, на деяких QPU `cx` є двокубітним вентилем за замовчуванням, а на інших — `ecr`. Дивись список можливих [рідних вентилів та операцій](/guides/qpu-information#native-gates) для отримання детальнішої інформації.

In [10]:
target["ecr"][(1, 0)]

InstructionProperties(duration=5.333333333333332e-07, error=0.007494257741828603)

![Вихідні дані попереднього блоку коду](../docs/images/guides/represent-quantum-computers/extracted-outputs/313e4743-0.svg)

Зауваж, що об'єкти `CXGate` були розкладені на вентилі `ecr` та однокубітні базові вентилі.
## Частоти помилок пристрою
Клас `Target` може містити інформацію про частоти помилок для операцій на пристрої.
Наприклад, наступний код отримує властивості для вентиля відлунного перехресного резонансу (ECR) між кубітами 1 та 0 (зауваж, що вентиль ECR є напрямленим):

In [11]:
from qiskit.transpiler import Target
from qiskit.circuit.controlflow import IfElseOp, SwitchCaseOp, ForLoopOp

err_targ = Target.from_configuration(
    basis_gates=basis_gates,
    coupling_map=coupling_map,
    num_qubits=target.num_qubits,
    custom_name_mapping={
        "if_else": IfElseOp,
        "switch_case": SwitchCaseOp,
        "for_loop": ForLoopOp,
    },
)

for i, (op, qargs) in enumerate(target.instructions):
    if op.name in basis_gates:
        err_targ[op.name][qargs] = target.instruction_properties(i)

Transpile with our new target `err_targ` as the target:

In [12]:
pass_manager = generate_preset_pass_manager(
    optimization_level=1, target=err_targ, seed_transpiler=12345
)
qc_t_cm_bg_et = pass_manager.run(qc)
qc_t_cm_bg_et.draw("mpl", idle_wires=False, fold=-1)

<Image src="../docs/images/guides/represent-quantum-computers/extracted-outputs/f1e270c4-e2cc-487e-a050-4180bc321b0b-0.svg" alt="Output of the previous code cell" />

Вихідні дані відображають тривалість вентиля (у секундах) та його частоту помилок. Щоб розкрити інформацію про помилки для транспілятора, побудуй цільову модель з `basis_gates` та `coupling_map` зверху і заповни її значеннями помилок з бекенду `FakeSherbrooke`.